# Лабораторна. GameScout: аналіз магазину відеоігор

**Тривалість:** 60 хвилин

**Контекст.** Ви — аналітик сервісу GameScout, який порівнює ціни на відеоігри. Команда хоче знати, скільки коштують ігри різних жанрів, чи впливає кількість жанрів на ціну і де шукати найвигідніші пропозиції. Для цього треба зібрати дані з каталогу інтернет-магазину.

**Сайт:** https://sandbox.oxylabs.io/products — тестовий магазин відеоігор, створений компанією Oxylabs спеціально для практики скрейпінгу.

Ця лабораторна побудована так само, як перша (BookScout), тож більшість прийомів вам уже знайомі. Нове тут:
- ціна записана в європейському форматі: `91,99 €` (кома замість крапки, знак валюти в кінці);
- у гри може бути **кілька жанрів**;
- каталог дуже великий, тому зберемо лише частину сторінок.

**План роботи:**

| Частина | Що робимо | Час |
|---|---|---|
| 0 | Підготовка | 2 хв |
| 1 | Розвідка сторінки | 8 хв |
| 2 | Дістаємо дані про одну гру | 15 хв |
| 3 | Збираємо кілька сторінок | 10 хв |
| 4 | Аналізуємо дані на чистому Python | 20 хв |
| 5 | Підсумок | 3 хв |

**Що здати:** цей ноутбук із заповненим кодом і відповідями в комірках «Ваша відповідь», а також файл `games.csv`.

**Оцінювання:** максимум 1.25 бала. Бали вказано біля кожного завдання. Якщо кілька завдань оцінюються разом, бали ставляться, лише коли виконано всі з них. Код, що падає з помилкою, і відповіді без пояснення не зараховуються.

## Частина 0. Підготовка (2 хв)

Функція `get_soup` уже готова. Вона об'єднує кроки, які в першій лабораторній ви писали щоразу:
1. `requests.get(url, timeout=10)` — завантажує сторінку (не чекаючи довше 10 секунд);
2. `raise_for_status()` — зупиняє програму з помилкою, якщо сервер відповів не кодом 200. Так ми одразу дізнаємося про проблему, а не будемо мовчки шукати дані на сторінці з помилкою;
3. `response.encoding = "utf-8"` — каже читати текст у правильному кодуванні (згадайте `Â£` з першої лабораторної);
4. `BeautifulSoup(...)` — повертає готовий `soup`, у якому можна шукати елементи.

Тепер замість чотирьох рядків достатньо написати `soup = get_soup(адреса)`.

In [ ]:
# !pip install requests beautifulsoup4

import csv
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

CATALOG_URL = "https://sandbox.oxylabs.io/products"

def get_soup(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    response.encoding = "utf-8"
    return BeautifulSoup(response.text, "html.parser")

print("Усе готово!")

## Частина 1. Розвідка сторінки (8 хв)

Відкрийте сторінку каталогу в браузері. Натисніть правою кнопкою на назву будь-якої гри → **Inspect**. Праворуч відкриється панель з HTML-кодом, і потрібний тег буде підсвічений.

**Як знаходити селектори:**
- Наводьте мишку на рядки в панелі HTML: відповідна частина сторінки підсвічуватиметься. Так легко знайти тег, який обгортає **всю картку** гри: піднімайтеся вгору по вкладених тегах, доки не підсвітиться вся картка.
- Для кожного елемента записуйте **назву тегу** і **клас** (атрибут `class`). Селектор складається з них через крапку: `<div class="price-wrapper">` → `div.price-wrapper`.
- Якщо в тегу немає класу, можна вказати лише назву тегу (`h4`), бо шукати ми будемо **всередині картки**, де такий тег один.

**Питання 1.1.** *(0.25 б. за 1.1 + 1.2)* Знайдіть і запишіть CSS-селектори для:
- картки однієї гри цілком;
- назви гри;
- ціни;
- жанрів гри;
- посилання на сторінку гри (зверніть увагу: у картці два посилання — на сторінку гри і кнопка «Add to Basket»; потрібне перше).

**Ваша відповідь:** 

**Завдання 1.2.** *(0.25 б. за 1.1 + 1.2)* Завантажте першу сторінку каталогу через `get_soup`, знайдіть усі картки ігор і виведіть їхню кількість. Перевірте, що вона збігається з кількістю ігор, які ви бачите в браузері.

Нагадування: `soup.select("селектор")` повертає **список** усіх знайдених елементів, тож його довжину можна дізнатися через `len()`. Якщо отримали `0` — селектор неправильний: поверніться до Inspect і перевірте назву тегу й класу.

In [ ]:
soup = ...   # TODO
cards = ...  # TODO
print(len(cards))

## Частина 2. Дістаємо дані про одну гру (15 хв)

Працюємо з першою карткою: `first = cards[0]`.

**Завдання 2.1.** *(0.25 б. за 2.1–2.3)* Отримайте ціну першої гри як `float`.

Ціна записана в європейському форматі: `91,99 €`. Щоб рахувати середню ціну чи сортувати, потрібне **число**, а текст `"91,99 €"` — це рядок. Функція `float()` розуміє лише формат `91.99`: знак `€` і кома призведуть до помилки `ValueError`.

Тому текст треба поступово «почистити». Методи рядків можна писати ланцюжком, один за одним, і кожен працює з результатом попереднього:
```
"91,99 €"  --replace("€", "")-->  "91,99 "  --strip()-->  "91,99"  --replace(",", ".")-->  "91.99"  --float()-->  91.99
```

⚠️ Не плутайте з першою лабораторною та магазином з `$1,099.00`: там кома розділяла **тисячі**, і її треба було **видалити**. Тут кома розділяє **копійки (центи)**, і її треба **замінити на крапку**. Якщо кому просто видалити, `91,99` перетвориться на `9199` — програма не видасть помилки, але всі ціни будуть у 100 разів більші.

Кроки:
1. Знайдіть у картці ціну й візьміть її текст.
2. Приберіть знак `€` і пробіли по краях.
3. Замініть кому на крапку.
4. Перетворіть на `float` і виведіть разом із `type(...)`.

In [ ]:
first = cards[0]

# TODO

**Завдання 2.2.** *(0.25 б. за 2.1–2.3)* Отримайте для першої гри список жанрів і їхню кількість.

**Жанри.** У гри може бути один жанр, а може бути кілька. Згадаймо різницю між двома методами пошуку:

| Метод | Що повертає | Коли використовувати |
|---|---|---|
| `select_one("...")` | **перший** знайдений елемент або `None` | коли елемент один (назва, ціна) |
| `select("...")` | **список** усіх знайдених елементів (може бути порожнім) | коли елементів може бути кілька (жанри) |

Список елементів ще не є списком назв жанрів: з кожного елемента треба взяти текст. Для цього зручно використати **list comprehension**:
```python
[елемент.text.strip() for елемент in список_елементів]
```
Читаємо так: «для кожного елемента зі списку візьми його текст без пробілів по краях і склади з цього новий список».

Кроки:
1. Знайдіть усі елементи жанрів у `first` через `select` і зберіть їхній текст у список `genres`.
2. Порахуйте кількість жанрів через `len()` і збережіть у `genres_count`.
3. Виведіть обидва значення. Для першої гри має вийти `['Action Adventure', 'Fantasy']` і `2`.

In [ ]:
# TODO: жанри

# TODO: кількість жанрів

**Завдання 2.3.** *(0.25 б. за 2.1–2.3)* Зберіть усе в функцію `parse_game(card)`, яка повертає словник:

| ключ | тип | приклад |
|---|---|---|
| `title` | str | `The Legend of Zelda: Ocarina of Time` |
| `price` | float | `91.99` |
| `genre` | str або None | `Action Adventure` (перший жанр зі списку) |
| `genres_count` | int | `2` |
| `url` | str | `https://sandbox.oxylabs.io/products/1` |

Два нові моменти:

**Перший жанр або `None`.** Якщо в гри немає жанрів, список порожній, і `genres[0]` викличе помилку `IndexError`. Тому спершу перевіряємо, чи список не порожній. Порожній список в умові `if` вважається `False`, а непорожній — `True`. Це можна записати в один рядок:
```python
genre = genres[0] if genres else None
```
Читаємо: «перший жанр, якщо список не порожній, інакше `None`».

**Повна адреса.** У `href` посилання записано лише `/products/1` — це **відносна** адреса: браузер сам дописує до неї адресу сайту, а наша програма — ні. Функція `urljoin` склеює їх правильно:
```python
urljoin("https://sandbox.oxylabs.io/products", "/products/1")   # 'https://sandbox.oxylabs.io/products/1'
```

Кроки:
1. Назву візьміть з тегу `h4`.
2. Ціну й жанри отримайте як у попередніх завданнях.
3. У `genre` запишіть перший жанр зі списку або `None` (як у прикладі вище), а в `genres_count` — довжину списку жанрів.
4. Знайдіть у картці посилання на сторінку гри (селектор з питання 1.1), візьміть його `href` і зробіть повну адресу через `urljoin(CATALOG_URL, href)`.
5. Поверніть словник з усіма п'ятьма ключами.
6. Перевірте функцію на першій і останній картці сторінки: значення мають збігатися з тим, що ви бачите в браузері.

In [ ]:
def parse_game(card):
    # TODO
    ...

print(parse_game(cards[0]))
print(parse_game(cards[-1]))

## Частина 3. Збираємо кілька сторінок (10 хв)

Номер сторінки записаний в адресі: натисніть унизу каталогу на кнопку «2» і подивіться на адресу:
```
https://sandbox.oxylabs.io/products?page=2
```
Частина після знака `?` називається **параметром запиту**: `page=2` означає «покажи другу сторінку». Як і в першій лабораторній, щоб перейти на іншу сторінку, досить змінити число в адресі. У Python адресу зручно зібрати f-рядком, підставивши номер у фігурні дужки: `f"{CATALOG_URL}?page={n}"`.

**Питання 3.1.** *(0.25 б. за 3.1 + 3.2)* Скільки всього сторінок у каталозі? Скільки часу займе збір **усіх** сторінок, якщо між запитами робити паузу 0.5 секунди? Чому в лабораторній ми зберемо лише перші 10 сторінок?

**Ваша відповідь:** 

**Завдання 3.2.** *(0.25 б. за 3.1 + 3.2)* Напишіть функцію `scrape_catalog(pages)`, яка збирає ігри з перших `pages` сторінок в один список.

Кроки:
1. Створіть порожній список `games`.
2. Напишіть цикл по номерах сторінок від 1 до `pages` включно.
3. Для кожного номера зберіть адресу f-рядком (`f"{CATALOG_URL}?page={n}"`), завантажте сторінку, розберіть усі картки через `parse_game` і додайте результати до `games` через `.extend()`.
4. Виведіть прогрес і зробіть паузу `time.sleep(0.5)`.
5. Поверніть `games`.

Нагадування з першої лабораторної:
- `parse_game` повертає словник для **однієї** гри, а на сторінці їх багато. Тому з кожної сторінки отримуємо **список** словників.
- `.append(x)` додає `x` як **один** елемент, тож додавши так список, ви отримаєте список у списку. `.extend(x)` додає **кожен елемент** списку `x` окремо — це те, що нам потрібно.
- Пауза `time.sleep(0.5)` потрібна, щоб не перевантажувати сервер: ним користуються й інші люди (і вся ваша група одночасно).

Спочатку перевірте на 2 сторінках (кількість ігор має бути вдвічі більшою, ніж у завданні 1.2), потім зберіть 10.

In [ ]:
def scrape_catalog(pages):
    # TODO
    ...

test = scrape_catalog(2)
print(len(test))

In [ ]:
games = scrape_catalog(10)
print(len(games))

## Частина 4. Аналізуємо дані на чистому Python (20 хв)

Тепер у нас є список `games`, і кожен його елемент — словник з даними однієї гри. Як і в першій лабораторній, аналізуємо дані за допомогою списків, словників і вбудованих функцій Python.

Нагадування прийомів з першої лабораторної:

| Що хочемо | Код |
|---|---|
| Список значень одного ключа | `[g["price"] for g in games]` |
| Сума / кількість / мін / макс | `sum(список)`, `len(список)`, `min(список)`, `max(список)` |
| Елемент з найбільшим значенням | `max(games, key=get_price)` |
| Підрахунок у словнику | `counts[ключ] = counts.get(ключ, 0) + 1` |
| Групування у словник зі списками | `groups.setdefault(ключ, []).append(значення)` |
| Фільтр | `[g for g in games if умова]` |
| Сортування | `sorted(список, key=функція, reverse=True)` |
| Перші N елементів | `список[:N]` |

**Нагадування про `key`.** Параметр `key` у `max`, `min` і `sorted` — це функція, яка каже, **за чим** порівнювати елементи. Функцію передаємо без дужок: `max` сам викличе її для кожного елемента.

In [ ]:
def get_price(game):
    return game["price"]

**Завдання 4.1.** *(0.25 б. за 4.1–4.4)* Перевірте, скільки ігор у списку `games` (має бути вдесятеро більше, ніж на одній сторінці). Зберіть усі ціни в список `prices` і виведіть середню ціну (округлену до двох знаків), мінімальну та максимальну.

In [ ]:
# TODO

**Ваша відповідь:** 

**Завдання 4.2.** *(0.25 б. за 4.1–4.4)* Виведіть назву й ціну найдорожчої та найдешевшої гри. Використайте `max` і `min` з параметром `key=get_price`.

In [ ]:
# TODO

**Завдання 4.3.** *(0.25 б. за 4.1–4.4)* Порахуйте, скільки ігор кожного жанру (ключ `genre`), і виведіть 5 найпоширеніших жанрів разом з кількістю ігор.

Кроки:
1. Порахуйте кількість ігор кожного жанру в словнику `genre_counts` (прийом «підрахунок у словнику»). Пропускайте ігри, у яких `genre` дорівнює `None`.
2. Відсортуйте **ключі** словника (назви жанрів) за кількістю ігор від більшої до меншої. Тут функцію для `key` писати не треба: метод словника `genre_counts.get` уже є функцією, яка за назвою жанру повертає кількість. Тому: `sorted(genre_counts, key=genre_counts.get, reverse=True)`.
3. Візьміть перші 5 жанрів і для кожного виведіть назву й кількість.

In [ ]:
# TODO

**Ваша відповідь:** 

**Завдання 4.4.** *(0.25 б. за 4.1–4.4)* Порахуйте середню ціну гри для кожного жанру й виведіть жанри від найдорожчого до найдешевшого.

Кроки:
1. Згрупуйте ціни за жанрами у словник `genre_prices`: ключ — жанр, значення — список цін (прийом «групування»). Ігри без жанру пропускайте.
2. Створіть новий словник `genre_avg`: ключ — жанр, значення — середня ціна його ігор, округлена до двох знаків. Для цього пройдіться циклом по `genre_prices`.
3. Відсортуйте жанри за середньою ціною (`key=genre_avg.get`) від більшої до меншої й виведіть кожен з його середньою ціною.

Який жанр найдорожчий у середньому? Чи можна довіряти цьому висновку, якщо в жанрі лише 1–2 гри (звірте з 4.3)?

In [ ]:
# TODO

**Ваша відповідь:** 

**Завдання 4.5.** *(0.25 б. за 4.5–4.7 + 5.1–5.2)* Скільки ігор мають 1 жанр, 2 жанри, 3 жанри (ключ `genres_count`)? Яка середня ціна ігор для кожної кількості жанрів? Чи дорожчі ігри, що належать до кількох жанрів?

Це ті самі два прийоми, що й у 4.3–4.4, лише групуємо не за жанром, а за кількістю жанрів. Виведіть результат по порядку: 1, 2, 3, ... (підказка: `sorted(словник)` сортує ключі).

In [ ]:
# TODO

**Ваша відповідь:** 

**Завдання 4.6.** *(0.25 б. за 4.5–4.7 + 5.1–5.2)* Знайдіть 5 найдешевших ігор **найпоширенішого** жанру (з 4.3), які належать до **кількох** жанрів (`genres_count` не менше 2).

Кроки:
1. Найпоширеніший жанр — перший елемент відсортованого списку з 4.3.
2. Відфільтруйте `games` за **двома** умовами одночасно: жанр дорівнює найпоширенішому **і** `genres_count` не менше 2. У list comprehension умови поєднуються словом `and`: `[g for g in games if умова1 and умова2]`.
3. Відсортуйте результат за ціною від меншої до більшої (`reverse` тут не потрібен).
4. Виведіть назву й ціну перших 5 ігор.

In [ ]:
# TODO

**Ваша відповідь:** 

**Завдання 4.7.** *(0.25 б. за 4.5–4.7 + 5.1–5.2)* Збережіть усі ігри у файл `games.csv` через `csv.DictWriter`, як у першій лабораторній. Колонки: `title`, `price`, `genre`, `genres_count`, `url`. Відкрийте файл і переконайтеся, що кількість рядків дорівнює кількості ігор плюс один рядок заголовка.

In [ ]:
# TODO

## Частина 5. Підсумок (3 хв)

**5.1.** *(0.25 б. за 4.5–4.7 + 5.1–5.2)* Ціна в першій лабораторній мала вигляд `£51.77`, а тут `91,99 €`. Що сталося б, якби ви застосували тут код з першої лабораторної (лише прибрали знак валюти)? Чому формат чисел важливо перевіряти на кожному новому сайті?

**5.2.** *(0.25 б. за 4.5–4.7 + 5.1–5.2)* Ми зберігали в ключі `genre` лише перший жанр. Яку інформацію ми при цьому втратили і як це могло вплинути на результати завдань 4.3–4.4?

**Ваша відповідь:** 

## Бонус (якщо залишився час) · не оцінюється

**Б.1.** Порахуйте популярність жанрів з урахуванням **усіх** жанрів кожної гри, а не лише першого. Для цього в `parse_game` збережіть повний список жанрів у ключ `genres`, а під час підрахунку пройдіться **вкладеним циклом** по всіх жанрах кожної гри. Чи змінився топ жанрів порівняно із завданням 4.3?

**Б.2.** Відкрийте сторінку будь-якої гри (посилання з ключа `url`). Які дані є на ній, але немає в картці каталогу? Напишіть функцію, яка дістає одне з таких полів, і застосуйте її до 5 найдорожчих ігор.

In [ ]:
# Бонус